# Redesigning Bad Charts

The explainer showed us how charts can lie without containing a single false number: truncated axes, dual-axis tricks, 3D distortions, inconsistent scales. We also learned that good design is not decoration but clarity — maximising data-ink, using colour accessibly, and matching the chart to its audience.

Now we get to apply that knowledge. A junior analyst has sent draft charts for the climate policy briefing. Several of them are misleading or confusing. Our job is to run each chart, diagnose what is wrong (and why it matters), and rebuild it properly before anyone important sees it.

By the end of this notebook we will be able to:

- Identify common chart design mistakes that mislead readers
- Explain **why** a chart is misleading, not just that it looks wrong
- Rebuild misleading charts into clear, honest alternatives

In [ ]:
%pip install -q pandas matplotlib seaborn

In [ ]:
%matplotlib inline

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

In [ ]:
temp = pd.read_csv("../data/global_temperature.csv")
energy = pd.read_csv("../data/energy_mix_by_country.csv")
co2 = pd.read_csv("../data/co2_emissions.csv")
renewables = pd.read_csv("../data/renewable_capacity.csv")

---

## Bad chart 1: Truncated y-axis

This was one of the first failure modes we met in the explainer. The junior analyst created this chart to show UK emissions declining. Run it and look carefully at the y-axis before reading the diagnosis below.

In [ ]:
# BAD CHART - run this cell
uk_co2 = co2[co2["country"] == "UK"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(uk_co2["year"], uk_co2["emissions_mt"], color="#c0392b", linewidth=2)
ax.set_ylim(300, 420)
ax.set_title("UK CO\u2082 Emissions Are Plummeting!", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Emissions (Mt)")
plt.tight_layout()
plt.show()

### What is wrong

The y-axis starts at 300 instead of 0. Because line charts encode values as position, this is technically permissible, but the analyst has used a **line** that reads visually as a **bar** — the filled area beneath the line implies length from a baseline. Combined with the sensational title, the chart makes it look like emissions have dropped by roughly half when the actual proportional change is much smaller.

Remember the rule from the explainer: bar charts encode values as lengths, and the length is accurate only when the axis starts at zero. A truncated axis on a bar chart is always misleading. On a line chart it can be acceptable, but only with clear labelling and a neutral title.

### The fix

Start the y-axis at zero. Use a neutral, factual title that lets the data speak.

In [ ]:
# FIXED CHART
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(uk_co2["year"], uk_co2["emissions_mt"], color="#c0392b", linewidth=2)
ax.set_ylim(0)
ax.set_title("UK CO\u2082 Emissions (2000\u20132024)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Emissions (Mt CO\u2082)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

With a zero baseline, the decline is visible but properly proportioned. The reader can judge the actual scale of the change without being nudged toward a conclusion.

**Exception**: truncated axes are acceptable when the values are naturally far from zero (e.g. stock prices, body temperatures). In those cases, starting at zero would compress the data into a flat line. The key is to be transparent: if we truncate, we make it obvious and keep the title honest.

---

## Bad chart 2: 3D pie chart

The explainer warned that 3D effects distort perception — slices at the front of a 3D pie chart appear larger than equally-sized slices at the back. The analyst has also combined this with too many categories and exploded slices. Run it and see for yourself.

In [ ]:
# BAD CHART - run this cell
from mpl_toolkits.mplot3d import Axes3D

china_2024 = energy[(energy["country"] == "China") & (energy["year"] == 2024)]

fig = plt.figure(figsize=(8, 6))

# Matplotlib does not have a true 3D pie, but the analyst
# simulated a 3D effect with shadow and exploded slices.
explode = (0.05, 0.05, 0.05, 0.05, 0.05, 0.05)
colours = ["#2c3e50", "#7f8c8d", "#f39c12", "#f1c40f", "#3498db", "#1abc9c"]

ax = fig.add_subplot(111)
wedges, texts, autotexts = ax.pie(
    china_2024["generation_twh"],
    labels=china_2024["source"],
    autopct="%1.1f%%",
    explode=explode,
    colors=colours,
    shadow=True,
    startangle=70,
)
ax.set_title("China Energy Mix 2024", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### What is wrong

Three problems, all covered in the explainer:

1. **Pie charts with many slices** are hard to read. Six slices with similar sizes make comparison nearly impossible because angle is a low-accuracy encoding.
2. **Exploded slices** and **shadows** are pure chartjunk. They add visual weight without adding information.
3. **Pie charts distort perception** even at their best. Humans are poor at comparing angles; we are much better at comparing lengths.

### The fix

Replace the pie chart with a horizontal bar chart. Sort by value so the largest source is immediately visible. This swaps the angle encoding for the most accurate one: position on a common scale.

In [ ]:
# FIXED CHART
china_sorted = china_2024.sort_values("generation_twh", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(china_sorted["source"], china_sorted["generation_twh"], color="#2980b9")
ax.set_title("China Energy Generation by Source (2024)", fontsize=14, fontweight="bold")
ax.set_xlabel("Generation (TWh)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

---

## Bad chart 3: Dual y-axes creating a false correlation

The explainer was blunt about dual axes: they can make any two variables look correlated by adjusting the scales until the lines overlap. The analyst has done exactly that here, plotting solar capacity and CO2 reductions on independently scaled axes to suggest a causal link.

In [ ]:
# BAD CHART - run this cell
uk_solar = renewables[renewables["country"] == "UK"][["year", "solar_gw"]]
uk_co2_sub = co2[(co2["country"] == "UK") & (co2["year"] >= 2010)][["year", "emissions_mt"]]

fig, ax1 = plt.subplots(figsize=(9, 5))

color1 = "#f39c12"
ax1.plot(uk_solar["year"], uk_solar["solar_gw"], color=color1, linewidth=2, label="Solar GW")
ax1.set_ylabel("Solar capacity (GW)", color=color1)
ax1.tick_params(axis="y", labelcolor=color1)
ax1.set_ylim(0, 25)

ax2 = ax1.twinx()
color2 = "#c0392b"
ax2.plot(uk_co2_sub["year"], uk_co2_sub["emissions_mt"], color=color2, linewidth=2, label="CO\u2082 Mt")
ax2.set_ylabel("CO\u2082 emissions (Mt)", color=color2)
ax2.tick_params(axis="y", labelcolor=color2)
ax2.set_ylim(250, 450)

ax1.set_title("Solar Growth Is Driving Down UK Emissions!", fontsize=14, fontweight="bold")
ax1.set_xlabel("Year")
plt.tight_layout()
plt.show()

### What is wrong

**Dual y-axes** with independently scaled ranges can make any two series look correlated. The analyst chose axis limits that make the lines cross at a visually dramatic point. The causal claim in the title is not supported by a chart alone.

Additional issues: the CO2 axis is truncated (starts at 250), and the title states a causal relationship when the chart only shows a coincidence in timing. This is precisely the kind of misleading design that erodes trust in analytical work.

### The fix

Use two separate panels. Each trend gets its own zero-based y-axis, so the reader can evaluate them independently without being led to a conclusion.

In [ ]:
# FIXED CHART
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(9, 7), sharex=True)

ax1.plot(uk_solar["year"], uk_solar["solar_gw"], color="#f39c12", linewidth=2)
ax1.set_ylabel("Solar capacity (GW)")
ax1.set_title("UK Solar Capacity and CO\u2082 Emissions (2010\u20132024)", fontsize=14, fontweight="bold")
ax1.set_ylim(0)
ax1.spines["top"].set_visible(False)
ax1.spines["right"].set_visible(False)

ax2.plot(uk_co2_sub["year"], uk_co2_sub["emissions_mt"], color="#c0392b", linewidth=2)
ax2.set_ylabel("CO\u2082 emissions (Mt)")
ax2.set_xlabel("Year")
ax2.set_ylim(0)
ax2.spines["top"].set_visible(False)
ax2.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

Two panels with a shared x-axis and independent, zero-based y-axes. The reader can compare the timing of both trends without being tricked by axis scaling. The neutral title presents the data; it does not push an interpretation.

---

## Bad chart 4: Rainbow colour map on sequential data

The explainer covered colour accessibility: rainbow colour maps have no perceptual ordering, they create false boundaries at sharp transitions, and they fail for readers with colour vision deficiency. The analyst has used one to encode temperature anomalies.

In [ ]:
# BAD CHART - run this cell
fig, ax = plt.subplots(figsize=(10, 5))

scatter = ax.scatter(temp["year"], temp["anomaly_c"],
                     c=temp["anomaly_c"], cmap="rainbow",
                     s=15, edgecolors="none")
plt.colorbar(scatter, ax=ax, label="Anomaly (\u00b0C)")
ax.set_title("Global Temperature Anomaly", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Anomaly (\u00b0C)")
plt.tight_layout()
plt.show()

### What is wrong

The **rainbow colour map** breaks several of the accessibility guidelines from the explainer:

1. It has no perceptual ordering. Yellow does not feel "between" green and red to most viewers.
2. It creates false boundaries where colours transition sharply (especially around green-to-yellow and yellow-to-red), making the reader see clusters that do not exist in the data.
3. It is not accessible to the roughly 8% of men with colour vision deficiency.

For data that goes from low to high, we should use a **sequential** colour map like `"viridis"`. For data centred on a meaningful zero point, we should use a **diverging** map like `"RdBu_r"` or `"coolwarm"`.

### The fix

Temperature anomaly is measured relative to a baseline, so zero is meaningful. A diverging colour map centred on zero communicates direction as well as magnitude.

In [ ]:
# FIXED CHART
fig, ax = plt.subplots(figsize=(10, 5))

vmax = max(abs(temp["anomaly_c"].min()), abs(temp["anomaly_c"].max()))
scatter = ax.scatter(temp["year"], temp["anomaly_c"],
                     c=temp["anomaly_c"], cmap="RdBu_r",
                     vmin=-vmax, vmax=vmax,
                     s=18, edgecolors="none")
plt.colorbar(scatter, ax=ax, label="Anomaly (\u00b0C)")
ax.axhline(y=0, color="grey", linestyle="--", linewidth=0.5)
ax.set_title("Global Temperature Anomaly (1850\u20132025)", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Anomaly (\u00b0C)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

The `RdBu_r` colour map is **diverging**: blue for below-zero anomalies, red for above-zero. The colour now communicates direction, not just magnitude. And because it relies on lightness differences as well as hue, it remains readable for most people with colour vision deficiency.

---

## Bad chart 5: Overcrowded bar chart

The explainer warned that too many categories make charts unreadable. The analyst tried to show the energy mix for all 10 countries in a single grouped bar chart. The result is a textbook example of a chart that tries to show everything and communicates nothing.

In [ ]:
# BAD CHART - run this cell
e_2024 = energy[energy["year"] == 2024]

countries = e_2024["country"].unique()
sources = e_2024["source"].unique()
x = np.arange(len(countries))
width = 0.12

fig, ax = plt.subplots(figsize=(12, 5))

for i, source in enumerate(sources):
    vals = []
    for c in countries:
        row = e_2024[(e_2024["country"] == c) & (e_2024["source"] == source)]
        vals.append(row["generation_twh"].values[0] if len(row) > 0 else 0)
    ax.bar(x + i * width, vals, width, label=source)

ax.set_xticks(x + width * 2.5)
ax.set_xticklabels(countries, rotation=45, ha="right", fontsize=8)
ax.set_title("Energy Generation by Source and Country (2024)", fontsize=14, fontweight="bold")
ax.set_ylabel("TWh")
ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

### What is wrong

With 10 countries and 6 sources, there are 60 bars crammed into one chart. The bars for smaller sources are invisible next to coal and gas. This is a data-ink problem: most of the ink on the chart is not helping the reader understand anything.

Think about audience here too. The briefing is for policymakers. They do not need to see every energy source for every country. They need a focused answer to a specific question.

### The fix

Choose a focused comparison. If the briefing is about renewable growth, show only renewables. Use a stacked bar to keep it readable and sort by total so the ranking is immediately clear.

In [ ]:
# FIXED CHART: focus on renewable sources only, stacked bar
renewable_sources = ["solar", "wind", "hydro"]
ren_2024 = e_2024[e_2024["source"].isin(renewable_sources)].copy()

pivot = ren_2024.pivot_table(index="country", columns="source",
                             values="generation_twh", fill_value=0)
pivot = pivot[renewable_sources]  # consistent order
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=True).index]  # sort by total

fig, ax = plt.subplots(figsize=(9, 5))
pivot.plot.barh(stacked=True, ax=ax,
               color=["#f1c40f", "#3498db", "#1abc9c"])
ax.set_title("Renewable Energy Generation by Country (2024)", fontsize=14, fontweight="bold")
ax.set_xlabel("Generation (TWh)")
ax.set_ylabel("")
ax.legend(title="Source")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
plt.show()

The stacked horizontal bar chart focuses on renewables, is sorted by total, and is readable at a glance. A policymaker can scan this in seconds and know which countries are generating the most renewable energy. That is the difference between a chart that dumps data and one that communicates.

---

## Exercises

Your turn. Each exercise gives you a bad chart. Run it, identify the problems using what we have learned from the explainer and the examples above, then write a corrected version. For each fix, think about which design principle you are applying: honest axes, appropriate encodings, colour accessibility, data-ink ratio, or audience awareness.

### Exercise 1: Fix the truncated axis

Run the cell below. The chart shows India's CO2 emissions. Identify what makes it misleading and write a corrected version.

In [ ]:
# BAD CHART - run this, then write your fix in the next cell
india_co2 = co2[co2["country"] == "India"]

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(india_co2["year"], india_co2["emissions_mt"], color="#e74c3c")
ax.set_ylim(900, 2600)
ax.set_title("India's Emissions Are Skyrocketing!", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
ax.set_ylabel("Emissions (Mt)")
plt.tight_layout()
plt.show()

In [ ]:
# Your code here


### Exercise 2: Fix the misleading dual axes

Run the cell below. It plots Germany's wind capacity and nuclear generation on dual y-axes to imply wind is replacing nuclear. Write a corrected version using two separate subplots.

In [ ]:
# BAD CHART - run this, then write your fix in the next cell
de_wind = renewables[renewables["country"] == "Germany"][["year", "wind_gw"]]
de_nuclear = energy[(energy["country"] == "Germany") & (energy["source"] == "nuclear")]

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(de_wind["year"], de_wind["wind_gw"], color="#3498db", linewidth=2)
ax1.set_ylabel("Wind (GW)", color="#3498db")
ax1.set_ylim(0, 100)

ax2 = ax1.twinx()
ax2.plot(de_nuclear["year"], de_nuclear["generation_twh"], color="#f39c12", linewidth=2)
ax2.set_ylabel("Nuclear (TWh)", color="#f39c12")
ax2.set_ylim(0, 200)

ax1.set_title("Wind Is Replacing Nuclear in Germany", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Your code here


### Exercise 3: Fix the rainbow colour map

Run the cell below. It uses a rainbow colour map to show per-capita emissions by country and year. Write a corrected version using an appropriate sequential colour map.

In [ ]:
# BAD CHART - run this, then write your fix in the next cell
fig, ax = plt.subplots(figsize=(10, 6))

sc = ax.scatter(co2["year"], co2["country"],
                c=co2["per_capita_t"], cmap="rainbow",
                s=60, edgecolors="none")
plt.colorbar(sc, ax=ax, label="Per-capita (t)")
ax.set_title("Per-Capita Emissions", fontsize=14, fontweight="bold")
ax.set_xlabel("Year")
plt.tight_layout()
plt.show()

In [ ]:
# Your code here


### Exercise 4: Fix the overcrowded chart

Run the cell below. It shows all energy sources for all countries in 2024 as a single pie chart. Write a corrected version that communicates the data clearly.

In [ ]:
# BAD CHART - run this, then write your fix in the next cell
all_2024 = energy[energy["year"] == 2024].copy()
all_2024["label"] = all_2024["country"] + " - " + all_2024["source"]

fig, ax = plt.subplots(figsize=(10, 10))
ax.pie(all_2024["generation_twh"], labels=all_2024["label"],
       startangle=90, textprops={"fontsize": 6})
ax.set_title("Global Energy Mix 2024", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Your code here


---

## Summary

We have practised the core skill of the explainer: spotting design choices that mislead and knowing how to fix them. Specifically, we:

- Identified **truncated y-axes** that exaggerate trends and restored honest baselines
- Replaced **3D pie charts** with readable bar charts that use position instead of angle
- Split **dual y-axis charts** into separate panels to prevent manufactured correlations
- Swapped **rainbow colour maps** for perceptually ordered, accessible alternatives
- Simplified **overcrowded charts** by focusing on a clear question rather than dumping all available data

The common thread: if a chart requires the reader to puzzle over it, the chart has failed. Our job as analysts is to make the data's message unmistakable.

In the next notebook, we will combine multiple charts into a single multi-panel dashboard, applying the layout principles from the final explainer.